##Imports

In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

##Data Pre-processing

In [46]:
##read the data
load_data = pd.read_csv("epl_final.csv")
test_data = pd.read_csv("epl-2025.csv")
future_matches = pd.read_csv("epl-2026.csv")

##remove rows with nan values
load_data = load_data.dropna()

##keep important columns
load_data = load_data[["HomeTeam", "AwayTeam", "FullTimeResult"]]
test_data = test_data[["HomeTeam", "AwayTeam", "FullTimeResult"]]
future_matches = future_matches[["HomeTeam", "AwayTeam"]]

def score_to_result(score):
    home, away = map(int, score.split("-"))

    if home > away:
        return 0      # Home win
    elif home == away:
        return 1      # Draw
    else:
        return 2      # Away win

test_data["FullTimeResult"] = test_data["FullTimeResult"].apply(score_to_result)



##target encodings
result_to_id = {
    "H": 0,
    "D": 1,
    "A": 2
}

##one hot encodings
home_encoded = pd.get_dummies(load_data["HomeTeam"], prefix="Home")
away_encoded = pd.get_dummies(load_data["AwayTeam"], prefix="Away")
home_encoded_test = pd.get_dummies(test_data["HomeTeam"], prefix="Home")
away_encoded_test = pd.get_dummies(test_data["AwayTeam"], prefix="Away")
home_encoded_future = pd.get_dummies(future_matches["HomeTeam"], prefix="Home")
away_encoded_future = pd.get_dummies(future_matches["AwayTeam"], prefix="Away")


X = pd.concat([home_encoded, away_encoded], axis=1)
X_test = pd.concat([home_encoded_test, away_encoded_test], axis=1)
X_future = pd.concat([home_encoded_future, away_encoded_future], axis=1)

X_test = X_test.reindex(columns=X.columns, fill_value=0) ##create columns which are in training set but not in test set
X_future = X_future.reindex(columns=X.columns, fill_value=0)
## map the data to encodings
load_data["FullTimeResult"] = load_data["FullTimeResult"].map(result_to_id)

Y = load_data["FullTimeResult"].values
Y_test = test_data["FullTimeResult"].values
print(X_test.shape)
print(X_test.dtypes.value_counts())
print(X_test.head())
X = X.astype(np.float32)
X_test = X_test.astype(np.float32)
X_future = X_future.astype(np.float32)

##convrt to tensor
X = torch.tensor(X.to_numpy(), dtype=torch.float32)
Y = torch.tensor(Y, dtype=torch.long)

X_test = torch.tensor(X_test.to_numpy(), dtype=torch.float32)
Y_test = torch.tensor(Y_test, dtype=torch.long)

X_future = torch.tensor(X_future.to_numpy(), dtype=torch.float32)



(380, 92)
int64    56
bool     36
Name: count, dtype: int64
   Home_Arsenal  Home_Aston Villa  Home_Birmingham  Home_Blackburn  \
0         False             False                0               0   
1         False              True                0               0   
2         False             False                0               0   
3         False             False                0               0   
4         False             False                0               0   

   Home_Blackpool  Home_Bolton  Home_Bournemouth  Home_Bradford  \
0               0            0             False              0   
1               0            0             False              0   
2               0            0             False              0   
3               0            0             False              0   
4               0            0             False              0   

   Home_Brentford  Home_Brighton  ...  Away_Southampton  Away_Stoke  \
0           False          False  ...        

##Multiregression implementation

In [47]:
##MultinomiallogisticRegressionmodel implementation
class MultinomiallogisticRegressionmodel(nn.Module):

  def __init__(self, input_size, num_classes):

    super(MultinomiallogisticRegressionmodel, self).__init__()
    self.linear = nn.Linear(input_size, num_classes)

  def forward(self, features):

    return self.linear(features)

  def compute_probabilities(self,x):

        logits = self.forward(x)
        probs = F.softmax(logits, dim=1)

        return probs

  def predict(self, x):

        probs = self.compute_probabilities(x)
        preds = torch.argmax(probs, dim=1)

        return preds






##Training

In [48]:

lr=0.05
num_epochs = 100
train_losses = []
input_size = X.shape[1]


model = MultinomiallogisticRegressionmodel(input_size,num_classes=3)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
Y=Y.long()
Y_test=Y_test.long()
for epoch in range(num_epochs):
  model.train()
  train_loss = 0
  train_total = 0
  outputs = model(X)
  loss = F.cross_entropy(outputs, Y)
  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  train_loss = train_loss+loss.item()*Y.size(0)

  train_total += Y.size(0)
  average_loss = train_loss / train_total
  print(f"Epoch {epoch+1}: Loss = {average_loss:.4f}")

  model.eval()
  test_loss = 0.0
  test_total = 0
  with torch.no_grad():
        test_outputs = model(X_test)
        test_loss = F.cross_entropy(test_outputs, Y_test)

        probabilities = torch.softmax(test_outputs, dim=1)
        predictions = torch.argmax(test_outputs, dim=1)
        accuracy = (predictions == Y_test).float().mean()

id_to_result = {
    0: "Home Win",
    1: "Draw",
    2: "Away Win"
}
print()
for i in range(len(predictions)):
    print(f"{test_data.iloc[i]['HomeTeam']} vs {test_data.iloc[i]['AwayTeam']}")
    print(f"Actual: {id_to_result[Y_test[i].item()]}")
    print(f"Predicted: {id_to_result[predictions[i].item()]}")
    print(f"Home Win Probability : {probabilities[i][0].item()*100:.3f}")
    print(f"Draw Probability     : {probabilities[i][1].item()*100:.3f}")
    print(f"Away Win Probability : {probabilities[i][2].item()*100:.3f}")
    print("-" * 40)

print(f"Test Loss: {test_loss.item():.4f}")
print(f"Test Accuracy: {accuracy:.4f}")






Epoch 1: Loss = 1.1000
Epoch 2: Loss = 1.0610
Epoch 3: Loss = 1.0356
Epoch 4: Loss = 1.0205
Epoch 5: Loss = 1.0116
Epoch 6: Loss = 1.0056
Epoch 7: Loss = 1.0004
Epoch 8: Loss = 0.9952
Epoch 9: Loss = 0.9903
Epoch 10: Loss = 0.9862
Epoch 11: Loss = 0.9834
Epoch 12: Loss = 0.9822
Epoch 13: Loss = 0.9825
Epoch 14: Loss = 0.9835
Epoch 15: Loss = 0.9845
Epoch 16: Loss = 0.9850
Epoch 17: Loss = 0.9848
Epoch 18: Loss = 0.9842
Epoch 19: Loss = 0.9834
Epoch 20: Loss = 0.9827
Epoch 21: Loss = 0.9822
Epoch 22: Loss = 0.9820
Epoch 23: Loss = 0.9820
Epoch 24: Loss = 0.9819
Epoch 25: Loss = 0.9818
Epoch 26: Loss = 0.9814
Epoch 27: Loss = 0.9808
Epoch 28: Loss = 0.9802
Epoch 29: Loss = 0.9798
Epoch 30: Loss = 0.9796
Epoch 31: Loss = 0.9795
Epoch 32: Loss = 0.9795
Epoch 33: Loss = 0.9795
Epoch 34: Loss = 0.9793
Epoch 35: Loss = 0.9792
Epoch 36: Loss = 0.9790
Epoch 37: Loss = 0.9788
Epoch 38: Loss = 0.9787
Epoch 39: Loss = 0.9786
Epoch 40: Loss = 0.9787
Epoch 41: Loss = 0.9787
Epoch 42: Loss = 0.9787
E

##2026/27 predictions

In [49]:
model.eval()
future_loss = 0.0
future_loss_total = 0
with torch.no_grad():
        future_outputs = model(X_future)
        ##future_loss = F.cross_entropy(test_outputs, Y_test)

        probabilities = torch.softmax(future_outputs, dim=1)
        predictions = torch.argmax(future_outputs, dim=1)
        ##accuracy = (predictions == Y_test).float().mean()
id_to_result = {
    0: "Home Win",
    1: "Draw",
    2: "Away Win"
}
print()
for i in range(len(predictions)):
    print(f"{future_matches.iloc[i]['HomeTeam']} vs {future_matches.iloc[i]['AwayTeam']}")
   ## print(f"Actual: {id_to_result[Y_test[i].item()]}")
    print(f"Predicted: {id_to_result[predictions[i].item()]}")
    print(f"Home Win Probability : {probabilities[i][0].item()*100:.3f}")
    print(f"Draw Probability     : {probabilities[i][1].item()*100:.3f}")
    print(f"Away Win Probability : {probabilities[i][2].item()*100:.3f}")
    print("-" * 40)


Arsenal vs Coventry
Predicted: Home Win
Home Win Probability : 82.367
Draw Probability     : 10.742
Away Win Probability : 6.891
----------------------------------------
Hull vs Man Utd
Predicted: Away Win
Home Win Probability : 27.057
Draw Probability     : 27.783
Away Win Probability : 45.160
----------------------------------------
Everton vs Crystal Palace
Predicted: Home Win
Home Win Probability : 47.627
Draw Probability     : 27.250
Away Win Probability : 25.123
----------------------------------------
Ipswich vs Sunderland
Predicted: Home Win
Home Win Probability : 46.213
Draw Probability     : 25.531
Away Win Probability : 28.257
----------------------------------------
Nott'm Forest vs Leeds
Predicted: Home Win
Home Win Probability : 43.958
Draw Probability     : 19.409
Away Win Probability : 36.633
----------------------------------------
Brentford vs Spurs
Predicted: Home Win
Home Win Probability : 39.722
Draw Probability     : 31.577
Away Win Probability : 28.701
---------